# Résultats - Détection de Fraudes & Cycles de Blanchiment
Lecture des résultats produits par les scripts Spark/GraphFrames depuis Delta Lake.

In [ ]:
import sys
sys.path[:0] = ['..', '../graph']
from config.config import DELTA_PATH
from config.spark_session import create_spark

spark = create_spark('_notebook')
print('Spark prêt')

## 1. Transactions ingérées en streaming (Kafka → Delta Lake)

In [ ]:
transactions = spark.read.format('delta').load(f'{DELTA_PATH}/transactions')
fraud_alerts = spark.read.format('delta').load(f'{DELTA_PATH}/fraud_alerts')

# Garde-fou : un rejeu du producteur ne doit pas gonfler les chiffres
KEY = ['step', 'type', 'amount', 'nameOrig', 'nameDest']
rows = transactions.count()
transactions = transactions.dropDuplicates(KEY).cache()
total = transactions.count()
frauds = fraud_alerts.dropDuplicates(KEY).count()
print(f'Lignes dans la table           : {rows:,}  ({rows - total:,} doublons)')
print(f'Transactions uniques ingerees  : {total:,}  ({total/6362620*100:.1f}% de PaySim)')
print(f'Fraudes uniques detectees      : {frauds:,}  ({frauds/total*100:.2f}%)')
transactions.groupBy('type').count().orderBy('count', ascending=False).show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 4)

tx_pd = transactions.groupBy('type').count().toPandas()
fig, axes = plt.subplots(1, 2)
tx_pd.sort_values('count', ascending=False).plot(kind='bar', x='type', y='count', ax=axes[0],
    color='steelblue', title='Transactions par type (streaming)', legend=False)
axes[0].set_xlabel(''); axes[0].tick_params(axis='x', rotation=45)
axes[1].bar(['Normales', 'Fraudes'], [total-frauds, frauds], color=['#4CAF50','#F44336'])
axes[1].set_title('Fraudes détectées en streaming')
for i, v in enumerate([total-frauds, frauds]):
    axes[1].text(i, v*0.95, f'{v:,}', ha='center', va='top', color='white', fontweight='bold')
plt.tight_layout(); plt.show()

## 2. Résultats GraphFrames - Noeuds suspects (PageRank, top 1 %)
Seuil = percentile 99 du score PageRank. Validation : part des comptes suspects réellement impliqués dans du blanchiment, comparée au taux de base.

In [ ]:
from build_graph import load_edges, vertices_of
from pyspark.sql.functions import col
try:
    suspicious = spark.read.format('delta').load(f'{DELTA_PATH}/suspicious_nodes')
    edges = load_edges(spark)
    n_total = vertices_of(edges).count()
    n_laund = vertices_of(edges.filter(col('is_laundering') == 1)).count()

    n_susp = suspicious.count()
    n_hits = suspicious.filter('is_laundering_account = 1').count()
    base, precision = n_laund / n_total, n_hits / n_susp
    print(f'Seuil PageRank (percentile 99)  : {suspicious.agg({"pagerank": "min"}).first()[0]:.4f}')
    print(f'Comptes suspects                : {n_susp:,} / {n_total:,} ({n_susp/n_total:.1%})')
    print(f'Dont impliques en blanchiment   : {n_hits:,} -> precision {precision:.2%}')
    print(f'Taux de base (tirage aleatoire) : {base:.2%} -> lift x{precision/base:.1f}')
    print(f'Rappel                          : {n_hits/n_laund:.2%} des {n_laund:,} comptes de blanchiment')
    suspicious.orderBy('pagerank', ascending=False).show(15, truncate=False)
except Exception as e:
    print(f'Résultats pas encore disponibles : {e}')
    print('Lancez graph/fraud_detection.py pour les générer')

## 3. Résultats GraphFrames - Cycles de blanchiment (SCC)

In [ ]:
try:
    cycle_nodes = spark.read.format('delta').load(f'{DELTA_PATH}/cycle_nodes')
    laundering_cycles = spark.read.format('delta').load(f'{DELTA_PATH}/laundering_cycles')
    print(f'Noeuds dans des cycles           : {cycle_nodes.count():,}')
    print(f'Transactions blanchiment en cycle: {laundering_cycles.count():,}')

    # Top cycles par taille
    cycle_nodes.groupBy('component').count().orderBy('count', ascending=False).show(10)
except Exception as e:
    print(f'Résultats pas encore disponibles : {e}')
    print('Lancez graph/cycle_detection.py pour les générer')

In [ ]:
try:
    cycles_pd = cycle_nodes.groupBy('component').count().orderBy('count', ascending=False).limit(20).toPandas()
    plt.figure(figsize=(10, 4))
    plt.bar(range(len(cycles_pd)), cycles_pd['count'], color='coral')
    plt.title('Taille des 20 plus grands cycles de blanchiment détectés (SCC)')
    plt.xlabel('Cycle (classé par taille)'); plt.ylabel('Nombre de comptes')
    plt.tight_layout(); plt.show()
except:
    print('Graphique disponible après cycle_detection.py')

## 4. Résumé final

In [ ]:
print('=' * 50)
print('RÉSUMÉ DU PROJET - DÉTECTION DE FRAUDES')
print('=' * 50)
print(f'Transactions PaySim uniques via Kafka  : {total:,}')
print(f'Fraudes detectees en streaming         : {frauds:,}')
try:
    print(f'Comptes suspects (PageRank top 1 %)    : {n_susp:,}  (lift x{precision/base:.1f})')
    print(f'Noeuds dans cycles de blanchiment      : {cycle_nodes.count():,}')
    print(f'Transactions blanchiment confirmées    : {laundering_cycles.count():,}')
except:
    print('(Graphe : lancer fraud_detection.py et cycle_detection.py)')
print('=' * 50)
spark.stop()